# Stage 0 — Data Cleaning

**Notebook:** `notebooks/01_data_cleaning.ipynb`  
**Plan:** `plan/stage00_data_cleaning.md`  
**Kernel:** `ml_env`

## Goals

1. Load `data/raw/Speed_Dating_Data.csv` (195-column original file).
2. Validate file integrity and confirm all columns required by Stages 1–5 are present.
3. Produce two flat parquet files:
   - `data/clean/cleaned.parquet` — main analysis set, Waves 6–9 excluded.
   - `data/clean/cleaned_all_waves.parquet` — all waves retained, for Stage 5 robustness only.
4. Write `data/clean/cleaning_report.json` for report writing and agent handoff.

## Key Design Decisions (do not override)

| Decision | Reason |
|----------|--------|
| Exclude Waves 6–9 from main analysis | Waves 1–5 & 10–21 use a constrained 100-point allocation; Waves 6–9 use independent 1–10 ratings — the two scales are incomparable |
| Output flat parquet (`index=False`) | `set_index(['iid','order'])` is deferred to Stage 3 (PanelOLS); flat output is compatible with both sklearn and statsmodels |
| Keep raw `race` column; no dummies here | Different models need different encoding strategies; premature dummy creation risks collinearity or duplicate columns |
| Do not drop rows with missing stated preferences | Normalization runs only on valid rows; Stage 1 calls `dropna(subset=stated_cols)` as needed |
| Use precise regex `(_2|_3|_s)$` to remove follow-up columns | Prevents accidentally dropping Stage 2 self-evaluation variables (`attr3_1`, `sinc3_1`, etc.) |


In [1]:
# ── Set working directory to project root ────────────────────────────────────
# Jupyter's cwd defaults to the notebook's own directory.
# We walk up to find the directory containing README.md (= project root)
# so that relative paths like data/raw/... resolve correctly from any launch point.
import os
from pathlib import Path

def _find_project_root(start: Path, marker: str = "README.md") -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(
        f"Could not find project root (no {marker} found above {start})"
    )

_project_root = _find_project_root(Path.cwd())
os.chdir(_project_root)
print(f"Working directory: {Path.cwd()}")

# ── Standard imports ──────────────────────────────────────────────────────────
from pathlib import Path
import json

import numpy as np
import pandas as pd

# Display settings: wide output for inspection
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

print(f"pandas: {pd.__version__}")
print(f"numpy:  {np.__version__}")


Working directory: /Users/chen/Study/UCB/STAT230a/speed-dating-halo


pandas: 2.3.3
numpy:  2.2.6


## Step 1 — Load and Validate the Raw File

In [2]:
# ── Path definitions ─────────────────────────────────────────────────────────
# IMPORTANT: use Speed_Dating_Data.csv (underscore), NOT speeddating.csv.
# speeddating.csv is a 123-column pre-processed version without iid/pid identifiers,
# which breaks Stages 2 and 3.
RAW_PATH  = Path("data/raw/Speed_Dating_Data.csv")
CLEAN_DIR = Path("data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# ── File existence check ──────────────────────────────────────────────────────
assert RAW_PATH.exists(), (
    f"Raw file not found: {RAW_PATH}\n"
    "Check that the filename is Speed_Dating_Data.csv (underscore, not space)."
)

# ── Load CSV ──────────────────────────────────────────────────────────────────
# encoding='latin-1' is required; the file contains non-ASCII characters that
# cause UnicodeDecodeError under the default utf-8 encoding.
df_raw = pd.read_csv(RAW_PATH, encoding="latin-1")

# ── Column count & key-column check ──────────────────────────────────────────
# 123 columns → wrong file (speeddating.csv pre-processed version)
# 195 columns → correct original file
assert df_raw.shape[1] == 195, (
    f"Expected 195 columns, got {df_raw.shape[1]}.\n"
    "You may have loaded speeddating.csv (123 cols) instead of the original."
)
assert {"iid", "pid", "wave", "order"}.issubset(df_raw.columns), (
    "Missing key identifier columns: iid / pid / wave / order"
)

print(f"Raw shape: {df_raw.shape}")
print(f"Waves present: {sorted(df_raw['wave'].unique())}")
print(f"Unique participants (iid): {df_raw['iid'].nunique()}")


Raw shape: (8378, 195)
Waves present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21)]
Unique participants (iid): 551


## Step 2 — Confirm All Pipeline Columns Are Present

In [3]:
# Every column listed here must exist in the raw file.
# A missing column here causes a silent KeyError downstream; we fail loudly instead.
required_for_pipeline = [
    # ── Outcome variables ────────────────────────────────────────────────────
    "dec",          # participant's yes/no decision (0/1)
    "match",        # mutual match (0/1)
    # ── Core partner ratings (post-date, 1–10 scale) ─────────────────────────
    "attr", "sinc", "intel", "fun", "amb", "shar",
    # ── Stated preference variables (pre-event 100-point allocation) ──────────
    "attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1",
    # ── Experimental structure ────────────────────────────────────────────────
    "iid",          # participant ID
    "pid",          # partner ID
    "wave",         # session wave (1–21)
    "order",        # within-session date order (time index for Stage 3 PanelOLS)
    "gender",       # 0 = female, 1 = male
    # ── Demographics ─────────────────────────────────────────────────────────
    "age",          # participant age
    "age_o",        # partner age (used to compute age_diff)
    "samerace",     # same-race pair (0/1)
    "int_corr",     # correlation of participant & partner 17-item interest vectors
    # ── Stage 2 extended feature set: self-evaluation variables ──────────────
    # NOTE: attr3_1 ends in "3_1", NOT "_3"; it is NOT a follow-up column
    # and must NOT be dropped by the follow-up regex filter.
    "attr3_1", "sinc3_1", "fun3_1", "intel3_1", "amb3_1",
    # ── Stage 2 extended feature set: partner ratings of self ────────────────
    "attr_o", "sinc_o", "intel_o", "fun_o", "amb_o", "shar_o",
]

missing_cols = [c for c in required_for_pipeline if c not in df_raw.columns]
assert not missing_cols, f"Missing required columns: {missing_cols}"

print(f"All {len(required_for_pipeline)} required pipeline columns present.")
print()

# Quick missing-rate preview for key variables (before any cleaning)
preview_cols = ["dec", "attr", "sinc", "intel", "fun", "amb", "shar",
                "attr1_1", "age", "income"]
print("Missing rates in raw data (key columns):")
print(df_raw[preview_cols].isna().mean().round(4).to_frame("missing_rate").T)


All 34 required pipeline columns present.

Missing rates in raw data (key columns):
                dec   attr   sinc  intel    fun    amb   shar  attr1_1    age  income
missing_rate 0.0000 0.0241 0.0331 0.0353 0.0418 0.0850 0.1274   0.0094 0.0113  0.4893


## Step 3 — Define the Cleaning Function

### Why Waves 6–9 Are Excluded from the Main Analysis

Waves 1–5 and 10–21 use a **constrained 100-point allocation** for stated preferences  
(`attr1_1`, `sinc1_1`, …): participants distribute exactly 100 points across six traits.

Waves 6–9 switched to **independent 1–10 ratings** per trait, with no sum constraint.

These two formats are fundamentally incomparable and cannot be mixed in the same regression.  
The main analysis excludes Waves 6–9. Stage 5 robustness checks include them after  
row-sum normalization to verify that conclusions hold.


In [4]:
# Verify the scale difference empirically using row-sum statistics.
# 100-point allocation waves: row sum ≈ 100 with std ≈ 0.
# Independent-rating waves:   row sum varies freely; std >> 0.

stated_cols_check = ["attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1"]

print("Stated-preference row-sum statistics by wave (mean ± std):")
print("-" * 55)
for w in [1, 2, 5, 6, 7, 8, 9, 10, 15, 21]:
    if w in df_raw["wave"].unique():
        sub = df_raw[df_raw["wave"] == w][stated_cols_check].dropna()
        if len(sub) > 0:
            s = sub.sum(axis=1)
            print(f"  Wave {w:2d}: mean={s.mean():6.1f}, std={s.std():5.1f}, n={len(s)}")

print()
print("Waves 1–5 & 10–21 : sum ≈ 100, std ≈ 0  →  100-point allocation (consistent)")
print("Waves 6–9          : sum varies, std > 0  →  independent 1–10 ratings (incomparable)")
print()
print("Decision: exclude Waves 6–9 from main analysis; retain for Stage 5 robustness.")


Stated-preference row-sum statistics by wave (mean ± std):
-------------------------------------------------------
  Wave  1: mean= 100.0, std=  0.0, n=200
  Wave  2: mean= 100.6, std=  3.5, n=592
  Wave  5: mean= 100.0, std=  0.0, n=170
  Wave  6: mean= 100.0, std=  0.0, n=45
  Wave  7: mean= 100.0, std=  0.0, n=512
  Wave  8: mean= 100.0, std=  0.0, n=200
  Wave  9: mean= 100.0, std=  0.0, n=800
  Wave 10: mean= 100.0, std=  0.0, n=162
  Wave 15: mean= 100.0, std=  0.0, n=684
  Wave 21: mean=  99.9, std=  2.3, n=946

Waves 1–5 & 10–21 : sum ≈ 100, std ≈ 0  →  100-point allocation (consistent)
Waves 6–9          : sum varies, std > 0  →  independent 1–10 ratings (incomparable)

Decision: exclude Waves 6–9 from main analysis; retain for Stage 5 robustness.


In [5]:
# ── Global constants shared across Steps 3–5 ─────────────────────────────────
stated_cols = ["attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1"]

# Core variables: missing values in any of these → drop the row.
# income has ~51% missing rate and is NOT included here; it stays as NaN.
core_vars = [
    "dec", "attr", "sinc", "intel", "fun", "amb", "shar",
    "iid", "pid", "wave", "gender", "order",
]


def basic_clean(df: pd.DataFrame, include_wave_6_9: bool = False):
    """
    Apply the standard cleaning pipeline to the raw DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Raw 195-column DataFrame loaded directly from Speed_Dating_Data.csv.
    include_wave_6_9 : bool
        False  →  main analysis set (Waves 6–9 excluded).
        True   →  all-waves set (Waves 6–9 retained and row-sum normalised).

    Returns
    -------
    df_clean : pd.DataFrame
        Cleaned flat DataFrame (no MultiIndex set).
    followup_cols : list[str]
        Names of dropped follow-up survey columns (printed for manual review).
    """
    df = df.copy()

    # ── 1. Parse income: comma-separated string → float ───────────────────────
    # Raw values look like "45,000" or the string "nan".
    # Step 1: astype("string") normalises Python None / np.nan → pd.NA.
    # Step 2: pd.to_numeric(errors="coerce") converts anything non-numeric to NaN.
    # Income has ~51% missing; we retain the column and never drop rows on it.
    df["income"] = (
        df["income"]
        .astype("string")
        .str.replace(",", "", regex=False)
    )
    df["income"] = pd.to_numeric(df["income"], errors="coerce")

    # ── 2. Wave filter ────────────────────────────────────────────────────────
    if not include_wave_6_9:
        before = len(df)
        df = df[~df["wave"].isin([6, 7, 8, 9])].copy()
        print(f"  Removed Waves 6–9: {before - len(df)} rows dropped, {len(df)} remaining.")

    # ── 3. Scale-type marker columns ──────────────────────────────────────────
    # These columns flag whether a row's stated-preference data came from a
    # 100-point allocation or an independent 1–10 scale. Useful for Stage 5
    # when all waves are loaded together.
    df["is_wave_6_9"] = df["wave"].isin([6, 7, 8, 9])
    df["stated_scale_type"] = np.where(
        df["is_wave_6_9"],
        "independent_1_10_normalized",   # Waves 6–9 after row-sum normalisation
        "100_point_allocation",           # All other waves: original constrained scale
    )

    # ── 4. Drop rows with missing core variables ───────────────────────────────
    # shar has 12.4% missing rate (largest among core vars) and causes the
    # most row loss (~1074 rows from the Wave-6–9-excluded set of 6816).
    before = len(df)
    df = df.dropna(subset=core_vars).copy()
    print(f"  Dropped rows with missing core vars: {before - len(df)}, {len(df)} remaining.")

    # ── 5. Normalise stated preferences ──────────────────────────────────────
    # Normalise only rows where all six stated-preference columns are present
    # and the row sum is positive (guards against division by zero).
    # Rows with any missing stated-preference value keep their NaN values;
    # Stage 1 will call dropna(subset=stated_cols) as needed.
    row_sums = df[stated_cols].sum(axis=1, min_count=len(stated_cols))
    # min_count=6: if even one column is missing, the sum returns NaN (not 0).
    valid_stated = row_sums.notna() & (row_sums > 0)
    df.loc[valid_stated, stated_cols] = (
        df.loc[valid_stated, stated_cols]
        .div(row_sums[valid_stated], axis=0)
        * 100
    )
    n_valid   = int(valid_stated.sum())
    n_invalid = int((~valid_stated).sum())
    print(f"  Stated preferences: {n_valid} rows normalised, "
          f"{n_invalid} rows retain NaN (Stage 1 will handle these).")

    # ── 6. Derived variables ──────────────────────────────────────────────────
    df["age_diff"] = (df["age"] - df["age_o"]).abs()   # absolute age difference

    # ── 7. Drop follow-up survey columns ─────────────────────────────────────
    # Follow-up columns end with _2, _3, or _s (e.g. attr_2, attr3_s).
    # They have 30–80% missing rates and are not used in any model.
    #
    # IMPORTANT distinction:
    #   attr3_1  →  self-evaluation variable needed by Stage 2; ends in "3_1" → NOT matched
    #   attr_3   →  follow-up column; ends in "_3"             → IS matched and dropped
    #
    # The regex (_2|_3|_s)$ matches only at the end of the column name.
    followup_mask = df.columns.str.contains(r"(_2|_3|_s)$", regex=True)
    followup_cols = df.columns[followup_mask].tolist()
    df = df.drop(columns=followup_cols)
    print(f"  Dropped {len(followup_cols)} follow-up columns.")
    print(f"  First 20: {followup_cols[:20]}")

    # ── 8. Uniqueness check for (iid, order) ──────────────────────────────────
    # PanelOLS in Stage 3 requires a unique (entity, time) index.
    # If duplicates exist, we print the offending rows for diagnosis
    # rather than silently deduplicating (which could mask data issues).
    dupes = df.duplicated(subset=["iid", "order"]).sum()
    if dupes > 0:
        print(f"WARNING: {dupes} duplicated (iid, order) rows found:")
        dup_rows = df[df.duplicated(subset=["iid", "order"], keep=False)]
        print(dup_rows[["iid", "order", "wave", "dec", "attr"]].head(20))
        raise AssertionError(
            f"Found {dupes} duplicated (iid, order) rows. Investigate before proceeding."
        )
    print(f"  (iid, order) uniqueness check passed.")

    return df, followup_cols


## Step 4 — Run the Cleaning Pipeline and Save Parquet Files

In [6]:
print("=" * 60)
print("Main analysis set (Waves 6–9 excluded)")
print("=" * 60)
df_main, dropped_main = basic_clean(df_raw, include_wave_6_9=False)

print()
print("=" * 60)
print("All-waves set (all waves retained, for Stage 5 only)")
print("=" * 60)
df_all, dropped_all = basic_clean(df_raw, include_wave_6_9=True)

# ── Save flat parquets ────────────────────────────────────────────────────────
# index=False: no MultiIndex is saved.
# set_index(['iid', 'order']) is performed in Stage 3 (Fixed Effects notebook).
# All downstream notebooks can load with pd.read_parquet() without any reset_index().
df_main.to_parquet(CLEAN_DIR / "cleaned.parquet",           index=False)
df_all.to_parquet( CLEAN_DIR / "cleaned_all_waves.parquet", index=False)

print()
print(f"Saved: {CLEAN_DIR / 'cleaned.parquet'}")
print(f"Saved: {CLEAN_DIR / 'cleaned_all_waves.parquet'}")
print()
print(f"Main shape:       {df_main.shape}")
print(f"All-waves shape:  {df_all.shape}")
print(f"All-waves > main: {len(df_all) > len(df_main)}")


Main analysis set (Waves 6–9 excluded)
  Removed Waves 6–9: 1562 rows dropped, 6816 remaining.
  Dropped rows with missing core vars: 1074, 5742 remaining.
  Stated preferences: 5640 rows normalised, 102 rows retain NaN (Stage 1 will handle these).
  Dropped 84 follow-up columns.
  First 20: ['attr1_s', 'sinc1_s', 'intel1_s', 'fun1_s', 'amb1_s', 'shar1_s', 'attr3_s', 'sinc3_s', 'intel3_s', 'fun3_s', 'amb3_s', 'satis_2', 'numdat_2', 'attr7_2', 'sinc7_2', 'intel7_2', 'fun7_2', 'amb7_2', 'shar7_2', 'attr1_2']
  (iid, order) uniqueness check passed.

All-waves set (all waves retained, for Stage 5 only)
  Dropped rows with missing core vars: 1347, 7031 remaining.
  Stated preferences: 6924 rows normalised, 107 rows retain NaN (Stage 1 will handle these).
  Dropped 84 follow-up columns.
  First 20: ['attr1_s', 'sinc1_s', 'intel1_s', 'fun1_s', 'amb1_s', 'shar1_s', 'attr3_s', 'sinc3_s', 'intel3_s', 'fun3_s', 'amb3_s', 'satis_2', 'numdat_2', 'attr7_2', 'sinc7_2', 'intel7_2', 'fun7_2', 'amb7_2',

/var/folders/ct/07v58nmj3f9fxfh48ks1b7w00000gn/T/ipykernel_83423/1932443052.py:99: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  followup_mask = df.columns.str.contains(r"(_2|_3|_s)$", regex=True)
/var/folders/ct/07v58nmj3f9fxfh48ks1b7w00000gn/T/ipykernel_83423/1932443052.py:99: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  followup_mask = df.columns.str.contains(r"(_2|_3|_s)$", regex=True)


## Step 5 — Sanity Checks

In [7]:
print("=" * 60)
print("SANITY CHECKS — main analysis set")
print("=" * 60)

# 1. Waves 6–9 fully excluded
bad_waves = df_main[df_main["wave"].isin([6, 7, 8, 9])]
assert len(bad_waves) == 0, f"Wave 6–9 rows still present: {len(bad_waves)}"
print(f"No Wave 6–9 rows. Waves in main set: {sorted(df_main['wave'].unique())}")

# 2. Core variables have zero missing values
missing_core = df_main[core_vars].isna().sum()
assert missing_core.sum() == 0, (
    f"Core vars have missing values:\n{missing_core[missing_core > 0]}"
)
print("All core variables: 0 missing values.")

# 3. Stage 2 self-evaluation columns were not accidentally dropped
self_eval_cols = ["attr3_1", "sinc3_1", "fun3_1", "intel3_1", "amb3_1"]
missing_self_eval = [c for c in self_eval_cols if c not in df_main.columns]
assert not missing_self_eval, (
    f"Self-eval columns incorrectly dropped: {missing_self_eval}"
)
print(f"Self-eval columns preserved: {self_eval_cols}")

# 4. income is float (may contain NaN)
assert pd.api.types.is_float_dtype(df_main["income"]), (
    f"income should be float, got {df_main['income'].dtype}"
)
income_missing_rate = df_main["income"].isna().mean()
print(f"income dtype: float. Missing rate: {income_missing_rate:.1%}")

# 5. Stated-preference normalisation: row sums ≈ 100 for valid rows
row_sums_after = df_main[stated_cols].sum(axis=1, min_count=len(stated_cols))
valid_rows = row_sums_after.notna()
n_valid_stated = int(valid_rows.sum())
max_err = float((row_sums_after[valid_rows] - 100).abs().max())
assert max_err < 0.01, (
    f"Stated-preference normalisation error too large: {max_err:.6f}"
)
print(f"Stated preferences: {n_valid_stated} valid rows, max row-sum error = {max_err:.2e}")
print(f"  ({int((~valid_rows).sum())} rows retain NaN stated preferences — Stage 1 handles these)")

# 6. (iid, order) uniqueness — redundant with the assert in basic_clean, kept as safety net
assert not df_main.duplicated(subset=["iid", "order"]).any(), (
    "(iid, order) not unique — Stage 3 PanelOLS will fail"
)
print("(iid, order) uniqueness confirmed — safe for Stage 3 PanelOLS.")

# 7. Key descriptive statistics
print()
print(f"  Rows:          {len(df_main):,}")
print(f"  Participants:  {df_main['iid'].nunique():,}")
print(f"  dec mean:      {df_main['dec'].mean():.4f}  (expected ≈ 0.42)")
print(f"  match mean:    {df_main['match'].mean():.4f}  (expected ≈ 0.17)")


SANITY CHECKS — main analysis set
No Wave 6–9 rows. Waves in main set: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21)]
All core variables: 0 missing values.
Self-eval columns preserved: ['attr3_1', 'sinc3_1', 'fun3_1', 'intel3_1', 'amb3_1']
income dtype: float. Missing rate: 51.1%
Stated preferences: 5640 valid rows, max row-sum error = 2.84e-14
  (102 rows retain NaN stated preferences — Stage 1 handles these)
(iid, order) uniqueness confirmed — safe for Stage 3 PanelOLS.

  Rows:          5,742
  Participants:  442
  dec mean:      0.4269  (expected ≈ 0.42)
  match mean:    0.1722  (expected ≈ 0.17)


## Step 6 — Write cleaning_report.json

`cleaning_report.json` serves three purposes:
- **Report writing**: key statistics (n_participants, dec_mean, etc.) can be cited directly in the Data section.
- **Reproducibility**: re-running this notebook should produce an identical JSON.
- **Agent handoff**: any downstream agent can confirm Stage 0 completed without re-executing the notebook.


In [8]:
report = {
    # ── Source file ───────────────────────────────────────────────────────────
    "raw_shape":   list(df_raw.shape),

    # ── Output shapes ─────────────────────────────────────────────────────────
    "main_shape":       list(df_main.shape),
    "all_waves_shape":  list(df_all.shape),

    # ── Participant counts ─────────────────────────────────────────────────────
    "n_participants_main":  int(df_main["iid"].nunique()),
    "n_participants_all":   int(df_all["iid"].nunique()),

    # ── Wave distribution ─────────────────────────────────────────────────────
    "main_wave_counts": {int(k): int(v) for k, v in
                         df_main["wave"].value_counts().sort_index().items()},
    "all_wave_counts":  {int(k): int(v) for k, v in
                         df_all["wave"].value_counts().sort_index().items()},

    # ── Key outcome statistics ────────────────────────────────────────────────
    "dec_mean_main":            float(df_main["dec"].mean()),
    "match_mean_main":          float(df_main["match"].mean()),
    "income_missing_rate_main": float(df_main["income"].isna().mean()),

    # ── Core-variable missing counts (all should be 0) ────────────────────────
    "core_missing_after_cleaning": {
        c: int(df_main[c].isna().sum()) for c in core_vars
    },

    # ── Cleaning operation summary ────────────────────────────────────────────
    "n_followup_cols_dropped":       len(dropped_main),
    "stated_valid_rows_main":        int(
        df_main[stated_cols].sum(axis=1, min_count=len(stated_cols)).notna().sum()
    ),
    "stated_max_normalization_error": float(
        (df_main[stated_cols]
         .sum(axis=1, min_count=len(stated_cols))
         .pipe(lambda s: (s[s.notna()] - 100).abs().max()))
    ),
}

report_path = CLEAN_DIR / "cleaning_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Saved: {report_path}")
print()
print(json.dumps(report, indent=2))


Saved: data/clean/cleaning_report.json

{
  "raw_shape": [
    8378,
    195
  ],
  "main_shape": [
    5742,
    114
  ],
  "all_waves_shape": [
    7031,
    114
  ],
  "n_participants_main": 442,
  "n_participants_all": 537,
  "main_wave_counts": {
    "1": 188,
    "2": 499,
    "3": 171,
    "4": 533,
    "5": 165,
    "10": 131,
    "11": 771,
    "12": 343,
    "13": 154,
    "14": 580,
    "15": 499,
    "16": 87,
    "17": 259,
    "18": 65,
    "19": 405,
    "20": 76,
    "21": 816
  },
  "all_wave_counts": {
    "1": 188,
    "2": 499,
    "3": 171,
    "4": 533,
    "5": 165,
    "6": 39,
    "7": 401,
    "8": 166,
    "9": 683,
    "10": 131,
    "11": 771,
    "12": 343,
    "13": 154,
    "14": 580,
    "15": 499,
    "16": 87,
    "17": 259,
    "18": 65,
    "19": 405,
    "20": 76,
    "21": 816
  },
  "dec_mean_main": 0.4268547544409613,
  "match_mean_main": 0.17223963775687914,
  "income_missing_rate_main": 0.511145942180425,
  "core_missing_after_cleaning": {
   

## Acceptance Criteria

In [9]:
# Run each criterion and collect pass/fail status.
# All must pass before Stage 0 is considered complete.
checks = {}

# 1. Raw file structure
checks["raw_195_cols"]         = df_raw.shape[1] == 195
checks["raw_key_cols_present"] = {"iid","pid","wave","order"}.issubset(df_raw.columns)

# 2. All pipeline columns present
checks["pipeline_cols_present"] = all(c in df_main.columns for c in required_for_pipeline)

# 3. Output files exist
checks["cleaned_parquet_exists"]      = (CLEAN_DIR / "cleaned.parquet").exists()
checks["all_waves_parquet_exists"]    = (CLEAN_DIR / "cleaned_all_waves.parquet").exists()
checks["cleaning_report_json_exists"] = (CLEAN_DIR / "cleaning_report.json").exists()

# 4. Waves 6–9 excluded from main set
checks["no_wave_6_9_in_main"] = not df_main["wave"].isin([6,7,8,9]).any()

# 5. All-waves set is larger than main set
checks["all_waves_larger"] = len(df_all) > len(df_main)

# 6. Main set row count in expected range
# After dropna on core vars (shar has 12.4% missing rate, the largest contributor),
# the observed count is ~5742.
checks["main_row_count_ok"] = 5500 <= len(df_main) <= 6100

# 7. Core variables have no missing values
checks["core_vars_no_missing"] = df_main[core_vars].isna().sum().sum() == 0

# 8. Stated-preference normalisation error within tolerance
_rs = df_main[stated_cols].sum(axis=1, min_count=len(stated_cols))
checks["stated_normalization_ok"] = (_rs[_rs.notna()] - 100).abs().max() < 0.01

# 9. (iid, order) uniqueness
checks["iid_order_unique"] = not df_main.duplicated(subset=["iid","order"]).any()

# 10. Outcome statistics within expected ranges
checks["dec_mean_range"]   = 0.38 <= df_main["dec"].mean() <= 0.46
checks["match_mean_range"] = 0.13 <= df_main["match"].mean() <= 0.20

# 11. income is float dtype
checks["income_is_float"] = pd.api.types.is_float_dtype(df_main["income"])

# 12. Stage 2 self-evaluation columns not accidentally removed
checks["self_eval_not_dropped"] = all(
    c in df_main.columns
    for c in ["attr3_1","sinc3_1","fun3_1","intel3_1","amb3_1"]
)

# ── Print results ─────────────────────────────────────────────────────────────
all_pass = True
print("Acceptance Criteria")
print("-" * 50)
for name, passed in checks.items():
    mark = "PASS" if passed else "FAIL"
    print(f"  [{mark}]  {name}")
    if not passed:
        all_pass = False

print()
if all_pass:
    print("All checks passed — Stage 0 complete.")
else:
    raise AssertionError("One or more checks failed — see above.")


Acceptance Criteria
--------------------------------------------------
  [PASS]  raw_195_cols
  [PASS]  raw_key_cols_present
  [PASS]  pipeline_cols_present
  [PASS]  cleaned_parquet_exists
  [PASS]  all_waves_parquet_exists
  [PASS]  cleaning_report_json_exists
  [PASS]  no_wave_6_9_in_main
  [PASS]  all_waves_larger
  [PASS]  main_row_count_ok
  [PASS]  core_vars_no_missing
  [PASS]  stated_normalization_ok
  [PASS]  iid_order_unique
  [PASS]  dec_mean_range
  [PASS]  match_mean_range
  [PASS]  income_is_float
  [PASS]  self_eval_not_dropped

All checks passed — Stage 0 complete.


## Data Snapshot (reference for Stage 1)

In [10]:
# Preview a few rows and summarise the cleaned dataset structure.
cols_preview = ["iid","pid","wave","order","gender","dec","attr","sinc","intel",
                "fun","amb","shar","attr1_1","age","age_diff","income"]

print("=== cleaned.parquet — first 3 rows ===")
print(df_main[cols_preview].head(3))
print()

print(f"=== Main set dimensions: {df_main.shape} ===")
print(f"  Numeric columns: {df_main.select_dtypes('number').shape[1]}")
print(f"  Object columns:  {df_main.select_dtypes('object').shape[1]}")
print()

print("=== Wave distribution (main set) ===")
print(df_main["wave"].value_counts().sort_index().to_frame("count").T)
print()

print("=== Gender distribution (0=female, 1=male) ===")
print(df_main["gender"].value_counts().sort_index())


=== cleaned.parquet — first 3 rows ===
   iid     pid  wave  order  gender  dec   attr   sinc  intel    fun    amb   shar  attr1_1     age  age_diff  \
0    1 11.0000     1      4       0    1 6.0000 9.0000 7.0000 7.0000 6.0000 5.0000  15.0000 21.0000    6.0000   
1    1 12.0000     1      3       0    1 7.0000 8.0000 7.0000 8.0000 5.0000 6.0000  15.0000 21.0000    1.0000   
2    1 13.0000     1     10       0    1 5.0000 8.0000 9.0000 8.0000 5.0000 7.0000  15.0000 21.0000    1.0000   

      income  
0 69487.0000  
1 69487.0000  
2 69487.0000  

=== Main set dimensions: (5742, 114) ===
  Numeric columns: 105
  Object columns:  8

=== Wave distribution (main set) ===
wave    1    2    3    4    5    10   11   12   13   14   15  16   17  18   19  20   21
count  188  499  171  533  165  131  771  343  154  580  499  87  259  65  405  76  816

=== Gender distribution (0=female, 1=male) ===
gender
0    2852
1    2890
Name: count, dtype: int64
